# Export a trained model to TFLite (CPU, no GPU needed)

Before running:
1. **Add Input → Notebooks** (tab) → pick the training notebook whose run finished, so its Output is attached.
2. **Accelerator: None**, **Internet: On**.

Produces `/kaggle/working/export/`: `detector.tflite` (FP16), `labels.txt`, `model_info.json`,
plus `results.csv` and a `training_summary.json` from the training run.

In [ ]:
import glob, os
cands = sorted(glob.glob("/kaggle/input/**/weights/best.pt", recursive=True))
print("\n".join(cands))
assert cands, "No best.pt found: attach the training notebook's output (Add Input -> Notebooks)"
BEST = cands[0]   # change if several runs are attached
RUN_DIR = os.path.dirname(os.path.dirname(BEST))
OUT = "/kaggle/working/export"
REPO = "https://github.com/divyanshkulshrestha23/road-anomaly-detection"
print("Using", BEST)
print(os.listdir(RUN_DIR))


In [ ]:
if not os.path.exists("/kaggle/working/road-anomaly-detection"):
    !git clone -q {REPO} /kaggle/working/road-anomaly-detection
else:
    !git -C /kaggle/working/road-anomaly-detection pull -q
%cd /kaggle/working/road-anomaly-detection
!pip install -q -r detector/requirements.txt
!git log --oneline -1


## Training curve and best epoch (from results.csv)

In [ ]:
import pandas as pd, json, shutil
os.makedirs(OUT, exist_ok=True)
df = pd.read_csv(f"{RUN_DIR}/results.csv")
df.columns = [c.strip() for c in df.columns]
shutil.copy2(f"{RUN_DIR}/results.csv", OUT)
key = next(c for c in df.columns if "mAP50(B)" in c and "95" not in c)
best = df.loc[df[key].idxmax()]
summary = {"epochs_run": int(df["epoch"].max()), "best_epoch": int(best["epoch"]),
           "best_val": {c: round(float(best[c]), 4) for c in df.columns if c.startswith("metrics/")},
           "last_val": {c: round(float(df.iloc[-1][c]), 4) for c in df.columns if c.startswith("metrics/")}}
for f in glob.glob(f"{os.path.dirname(RUN_DIR)}/../*summary*.json") + glob.glob(f"/kaggle/input/**/*summary*.json", recursive=True):
    shutil.copy2(f, OUT)
with open(f"{OUT}/training_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
print(df[["epoch", key]].tail(10).to_string(index=False))


## Export FP16 TFLite

In [ ]:
!python detector/export_tflite.py --weights {BEST} --out {OUT}

In [ ]:
shutil.copy2(BEST, OUT)
shutil.rmtree("/kaggle/working/road-anomaly-detection", ignore_errors=True)
print(sorted(os.listdir(OUT)))
